# Leitos hospitalares para sarampo — CNES 2007 a 2026

Identifica, nos arquivos `dados/cnes/leitos_cnes_AAAA.csv` (CNES/DATASUS, leitos por UF × competência × tipo × especialidade),
os leitos que podem ser destinados a sarampo e mostra a quantidade existente e a disponível ao SUS ao longo do tempo.

## O que os dados permitem afirmar

- O CNES **não tem tipo de leito "sarampo"** nem "infectologia/DIP". A tabela oficial de especialidades de leito (CnesWeb, `Mod_Ind_Tipo_Leito`) foi lida
  integralmente (64 códigos) e o único tipo voltado a doença transmissível é **Unidade de Isolamento** (`TP_LEITO=3`, `CODLEITO=66`).
- Por isso os leitos de sarampo são identificados **pelo uso**, no SIH (seção abaixo), e quantificados no CNES pelos grupos correspondentes. A Unidade de
  Isolamento é mostrada à parte como capacidade possível: o CNES não informa qual doença ocupa cada leito, e o isolamento serve a qualquer doença transmissível.
- `QT_EXIST` = leitos existentes; `QT_SUS` = leitos existentes disponibilizados ao SUS. **Ocupação não está no CNES** e não há ocupação de leitos de
  isolamento nos dados do projeto; portanto "disponíveis" significa *capacidade cadastrada*, não leitos livres.
- No SIH, `ESPEC` mostra o leito **observado** em cada internação de sarampo: Pediátrico (07) e Clínico (03) somam 99,3%; 114 de 3.610 passaram por UTI
  (`MARCA_UTI`). Esses são os leitos efetivamente usados; o isolamento é uma capacidade cadastrada que o SIH não permite confirmar como usada (não há
  campo de isolamento na internação).
- Leitos gerais usados como referência (Clínica geral `2/33`, Pediatria clínica `5/45`) são mostrados à parte e **não são exclusivos de sarampo**.

Fonte da tabela de códigos: https://cnes2.datasus.gov.br/Mod_Ind_Tipo_Leito.asp?VEstado=00

In [ ]:
# [1] Imports, leitura e tabela de códigos
from pathlib import Path
import warnings
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.1f}".format
mostrar = display if "get_ipython" in globals() else (lambda d: print(d.to_string()))

BASE = next(p for p in (Path("dados/cnes"), Path("../dados/cnes"), Path("cnes")) if p.exists())

# Leitos de interesse: (TP_LEITO, CODLEITO)
ISOLAMENTO = ("3", "66")                   # Unidade de Isolamento
REFERENCIAS = {("2", "33"): "Clínica geral (2/33)", ("5", "45"): "Pediatria clínica (5/45)"}

# Nomes conforme CnesWeb (tabela de tipos de leito). Códigos fora dela aparecem como "não consta".
NOMES = {
 ("1","01"):"Buco maxilo facial",("1","02"):"Cardiologia (cirúrgico)",("1","03"):"Cirurgia geral",("1","04"):"Endocrinologia",
 ("1","05"):"Gastroenterologia",("1","06"):"Ginecologia",("1","08"):"Nefrologia/urologia",("1","09"):"Neurocirurgia",
 ("1","11"):"Oftalmologia",("1","12"):"Oncologia (cirúrgico)",("1","13"):"Ortopedia/traumatologia",("1","14"):"Otorrinolaringologia",
 ("1","15"):"Plástica",("1","16"):"Torácica",("1","67"):"Transplante",("1","90"):"Queimado adulto (cirúrgico)",("1","91"):"Queimado pediátrico (cirúrgico)",
 ("2","31"):"AIDS",("2","32"):"Cardiologia (clínico)",("2","33"):"Clínica geral",("2","35"):"Dermatologia",("2","36"):"Geriatria",
 ("2","37"):"Hansenologia",("2","38"):"Hematologia",("2","40"):"Nefrourologia",("2","41"):"Neonatologia",("2","42"):"Neurologia",
 ("2","44"):"Oncologia (clínico)",("2","46"):"Pneumologia",("2","87"):"Saúde mental",("2","88"):"Queimado adulto (clínico)",("2","89"):"Queimado pediátrico (clínico)",
 ("4","10"):"Obstetrícia cirúrgica",("4","43"):"Obstetrícia clínica",("5","45"):"Pediatria clínica",("5","68"):"Pediatria cirúrgica",
 ("6","34"):"Crônicos",("6","47"):"Psiquiatria",("6","48"):"Reabilitação",("6","49"):"Pneumologia sanitária",("6","84"):"Acolhimento noturno",
 ("7","07"):"Hospital-dia cirúrgico/diagnóstico/terapêutico",("7","69"):"Hospital-dia AIDS",("7","70"):"Hospital-dia fibrose cística",
 ("7","71"):"Hospital-dia intercorrência pós-transplante",("7","72"):"Hospital-dia geriatria",("7","73"):"Hospital-dia saúde mental",
 ("3","74"):"UTI adulto I",("3","77"):"UTI pediátrica I",
 ("3","65"):"Unidade intermediária neonatal",("3","66"):"UNIDADE DE ISOLAMENTO",("3","75"):"UTI adulto II",("3","76"):"UTI adulto III",
 ("3","78"):"UTI pediátrica II",("3","79"):"UTI pediátrica III",("3","80"):"UTI neonatal I",("3","81"):"UTI neonatal II",("3","82"):"UTI neonatal III",
 ("3","83"):"UTI queimados",("3","85"):"UCO II",("3","86"):"UCO III",("3","92"):"Cuidados intermediários neonatal convencional",
 ("3","93"):"Cuidados intermediários neonatal canguru",("3","94"):"Cuidados intermediários pediátrico",("3","95"):"Cuidados intermediários adulto",
 ("3","96"):"Suporte ventilatório pulmonar",
}

arqs = sorted(BASE.glob("leitos_cnes_20??.csv"))
leitos = pd.concat((pd.read_csv(a, sep=";", dtype=str) for a in arqs), ignore_index=True)
leitos[["QT_EXIST", "QT_SUS"]] = leitos[["QT_EXIST", "QT_SUS"]].astype(int)
leitos["data"] = pd.to_datetime(leitos["COMPETEN"], format="%Y%m")
print(f"{len(arqs)} arquivos | {len(leitos):,} linhas | {leitos.data.min():%m/%Y} a {leitos.data.max():%m/%Y} | {leitos.UF.nunique()} UFs")

## Identificação dos leitos

Todos os pares (`TP_LEITO`, `CODLEITO`) presentes na última competência, com o nome oficial e o total do Brasil.
O conjunto adotado como leitos de sarampo está marcado.

In [ ]:
# [2] Catálogo de todos os tipos de leito presentes nos dados
ult = leitos[leitos.data == leitos.data.max()]
cat = ult.groupby(["TP_LEITO", "CODLEITO"])[["QT_EXIST", "QT_SUS"]].sum().reset_index()
cat["nome"] = [NOMES.get((t, c), "não consta na tabela CnesWeb") for t, c in zip(cat.TP_LEITO, cat.CODLEITO)]
cat["adotado"] = ["SIM" if (t, c) == ISOLAMENTO else "" for t, c in zip(cat.TP_LEITO, cat.CODLEITO)]
print(f"Última competência: {leitos.data.max():%m/%Y}")
mostrar(cat.style.format({"QT_EXIST": "{:,.0f}", "QT_SUS": "{:,.0f}"}) if "get_ipython" in globals() else cat)

print("\nTipos cujo nome sugere isolamento/doença infecciosa:",
      [f"{t}/{c} {n}" for (t, c), n in NOMES.items() if any(k in n.lower() for k in ("isola", "infec", "sarampo"))])

## Quais leitos internam sarampo de fato? (SIH)

O SIH (`dados/sih/internacoes_sarampo_sih_AAAA.csv`) tem as internações com CID B05 (sarampo). O único campo que diz **em que tipo de leito** o paciente ficou é
`MARCA_UTI`, e só para quem passou por UTI. Os códigos de `MARCA_UTI` (esquema TAB_SIH, a partir de jan/2008) são os mesmos números de `CODLEITO` das UTIs
no CNES, o que permite ligar a internação ao leito (correspondência adotada em `parametros/calcular_ocupacao_uti.py`).

Os CSVs agora trazem também `ESPEC` (especialidade do leito da AIH) e `CNES` (hospital), campos 4 e 75 do layout oficial do SIH
(`IT_SIHSUS_1603.pdf`). Com eles o tipo de leito é conhecido para **todas** as internações, não só as de UTI.

In [ ]:
# [2b] Internações de sarampo no SIH e tipos de UTI usados
sih = pd.concat((pd.read_csv(a, sep=";", dtype=str) for a in sorted(BASE.parent.glob("sih/internacoes_sarampo_sih_20??.csv"))), ignore_index=True)
sih["DIAS_PERM"] = sih["DIAS_PERM"].astype(int)
sih["ano"] = sih["COMPETEN"].str[:4].astype(int)
print(f"{len(sih):,} internações de sarampo ({sih.ano.min()}-{sih.ano.max()}) | colunas: {', '.join(sih.columns[:11])}, ...")
print("Colunas de tipo de leito/hospital no CSV:", [c for c in ("ESPEC", "CNES") if c in sih.columns] or "nenhuma")

uti = sih[sih.MARCA_UTI != "00"]
print(f"Passaram por UTI: {len(uti)} ({100*len(uti)/len(sih):.1f}%) | sem UTI (leito geral, ver ESPEC):{len(sih)-len(uti):,}")

# mesma numeração do CODLEITO do CNES (TP_LEITO=3)
uso = (uti.groupby("MARCA_UTI").size().rename("internacoes").to_frame()
          .assign(leito_cnes=lambda d: [NOMES.get(("3", c), "não consta") for c in d.index]))
uso["pct_das_UTI"] = 100 * uso.internacoes / uso.internacoes.sum()
mostrar(uso)

### Especialidade do leito da internação (`ESPEC`)

`ESPEC` registra a especialidade do leito de **todas** as internações. Nomes com fonte: `03 = Clínico` e `07 = Pediátrico`
(tabela de especialidade da AIH, Portaria SAES; [CONASS](https://www.conass.org.br/?p=22171)). Os demais códigos (01, 02, 09) somam
poucas internações e ficam sem nome, pois não achei a fonte da tabela completa.

In [ ]:
# [2c] Tipo de leito por especialidade da AIH (ESPEC)
ESPEC_NOME = {"03": "Clínico", "07": "Pediátrico"}          # nomes com fonte (Portaria SAES / CONASS)
assert "ESPEC" in sih.columns, "Gere os CSVs com ESPEC: python parametros/exportar_datasus.py --bases sih"

por_espec = (sih.groupby("ESPEC").agg(internacoes=("N_AIH", "size"), dias_medios=("DIAS_PERM", "mean"),
                                       hospitais=("CNES", "nunique"), com_UTI=("MARCA_UTI", lambda m: (m != "00").sum())))
por_espec.insert(0, "leito", [ESPEC_NOME.get(e, "sem nome na fonte") for e in por_espec.index])
por_espec["pct"] = 100 * por_espec.internacoes / por_espec.internacoes.sum()
mostrar(por_espec.sort_values("internacoes", ascending=False))
print(f"Hospitais (CNES) distintos com internação de sarampo: {sih.CNES.nunique()}")

ano_espec = sih.groupby(["ano", "ESPEC"]).size().unstack(fill_value=0)
ano_espec.columns = [f"{c} {ESPEC_NOME.get(c, '')}".strip() for c in ano_espec.columns]
ax = ano_espec.plot.bar(stacked=True, figsize=(11, 4.5), width=.85)
ax.set(title="Internações de sarampo por especialidade do leito (SIH)", ylabel="Internações", xlabel="Ano de competência")
ax.grid(axis="y", alpha=.3); plt.tight_layout(); plt.show()

### Leitos Clínicos e Pediátricos no CNES

`ESPEC` 03 e 07 correspondem, pelo nome, aos grupos de leitos do CNES `TP_LEITO=2` (clínico) e `TP_LEITO=5` (pediátrico; Pediatria clínica `5/45` e
cirúrgica `5/68`). Esses dois grupos concentram 99,3% das internações de sarampo. **A ligação é por nome da especialidade**: o SIH não informa o `CODLEITO`
(Clínica geral, Pneumologia, etc.) dentro do grupo clínico.

In [ ]:
# [2e] Leitos existentes e SUS nos grupos usados por sarampo (ESPEC 03 -> TP_LEITO 2, ESPEC 07 -> TP_LEITO 5)
GRUPOS = {"2": "Clínico (ESPEC 03)", "5": "Pediátrico (ESPEC 07)"}
g = leitos[leitos.TP_LEITO.isin(GRUPOS)].groupby(["data", "TP_LEITO"])[["QT_EXIST", "QT_SUS"]].sum().unstack("TP_LEITO")
g.columns = [f"{GRUPOS[t]} — {'existentes' if q == 'QT_EXIST' else 'SUS'}" for q, t in g.columns]
mostrar(g.iloc[::12])

fig, ax = plt.subplots(figsize=(11, 4.5))
for c in g.columns:
    ax.plot(g.index, g[c], label=c, lw=2, ls="-" if "existentes" in c else "--")
ax.set(title="Leitos clínicos e pediátricos — Brasil (CNES)", ylabel="Leitos"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

### Quantidade de leitos de UTI usados por sarampo ao longo do tempo

UTI adulto e pediátrica (códigos 75, 76, 78 e 79 no SIH) respondem por **112 das 114** internações de sarampo em UTI; as 2 restantes foram em UTI neonatal.
Soma dos leitos de CNES desses códigos (mesmos grupos de `calcular_leitos_uti.py`).

In [ ]:
# [2d] Leitos de UTI adulto + pediátrica (existentes e SUS), Brasil
UTI_SARAMPO = {"adulto": ["61", "74", "75", "76"], "pediatrica": ["62", "77", "78", "79"]}
cods = [c for v in UTI_SARAMPO.values() for c in v]
u = leitos[(leitos.TP_LEITO == "3") & leitos.CODLEITO.isin(cods)].copy()
u["grupo"] = u.CODLEITO.map({c: g for g, v in UTI_SARAMPO.items() for c in v})
uti_leitos = u.groupby(["data", "grupo"])[["QT_EXIST", "QT_SUS"]].sum().unstack("grupo")
uti_leitos.columns = [f"{g}_{'existentes' if q == 'QT_EXIST' else 'sus'}" for q, g in uti_leitos.columns]
uti_leitos["total_existentes"] = uti_leitos.filter(like="_existentes").sum(axis=1)
uti_leitos["total_sus"] = uti_leitos.filter(like="_sus").sum(axis=1)
mostrar(uti_leitos.iloc[::12][["adulto_existentes", "pediatrica_existentes", "total_existentes", "total_sus"]])

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(uti_leitos.index, uti_leitos.total_existentes, label="Existentes", color="#1f4e79", lw=2)
ax.plot(uti_leitos.index, uti_leitos.total_sus, label="SUS", color="#c0392b", lw=2)
ax.set(title="Leitos de UTI adulto + pediátrica — Brasil (CNES)", ylabel="Leitos"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

## Referência: leitos de isolamento ao longo do tempo (Brasil)

In [ ]:
# [3] Série mensal do Brasil — Unidade de Isolamento
def serie(tp, cod):
    x = leitos[(leitos.TP_LEITO == tp) & (leitos.CODLEITO == cod)]
    return x.groupby("data")[["QT_EXIST", "QT_SUS"]].sum().rename(columns={"QT_EXIST": "existentes", "QT_SUS": "sus"})

iso = serie(*ISOLAMENTO)
iso["nao_sus"] = iso.existentes - iso.sus

esperado = (leitos.data.max().year - 2007) * 12 + leitos.data.max().month
print(f"Meses na série: {len(iso)} (esperado {esperado})")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(iso.index, iso.existentes, label="Existentes", color="#1f4e79", lw=2)
ax.plot(iso.index, iso.sus, label="Disponíveis ao SUS", color="#c0392b", lw=2)
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2022-12-31"), color="grey", alpha=.12, label="mar/2020 a dez/2022")
ax.set(title="Leitos de Unidade de Isolamento — Brasil (CNES)", ylabel="Leitos", xlabel="Competência")
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

anual = iso.groupby(iso.index.year).agg(existentes_dez=("existentes", "last"), sus_dez=("sus", "last"),
                                        existentes_media=("existentes", "mean"), sus_media=("sus", "mean"))
anual["pct_sus_dez"] = 100 * anual.sus_dez / anual.existentes_dez
print("Último mês de cada ano (2026: última competência) e média anual:")
mostrar(anual)

## Distribuição por UF

Leitos de isolamento por UF na última competência e evolução nas 5 UFs com mais leitos.

In [ ]:
# [4] Por UF
x = leitos[(leitos.TP_LEITO == ISOLAMENTO[0]) & (leitos.CODLEITO == ISOLAMENTO[1])]
uf = x.pivot_table(index="data", columns="UF", values="QT_EXIST", aggfunc="sum").fillna(0)
uf_sus = x.pivot_table(index="data", columns="UF", values="QT_SUS", aggfunc="sum").fillna(0)

ultimo = pd.DataFrame({"existentes": uf.iloc[-1], "sus": uf_sus.iloc[-1]}).astype(int).sort_values("existentes", ascending=False)
ultimo["pct_do_brasil"] = 100 * ultimo.existentes / ultimo.existentes.sum()
print(f"Competência {uf.index[-1]:%m/%Y}:")
mostrar(ultimo)

fig, ax = plt.subplots(figsize=(11, 4.5))
for u in ultimo.index[:5]:
    ax.plot(uf.index, uf[u], label=u, lw=1.8)
ax.set(title="Leitos de isolamento existentes — 5 maiores UFs", ylabel="Leitos")
ax.grid(alpha=.3); ax.legend(ncol=5); plt.tight_layout(); plt.show()

## Referência: leitos gerais (não exclusivos de sarampo)

Clínica geral e Pediatria clínica, para comparar a evolução do isolamento com a dos leitos gerais. Os dados do CNES não dizem quantos desses
leitos atendem sarampo.

In [ ]:
# [5] Isolamento vs. leitos gerais (leitos existentes; índice jan/2007 = 100)
comp = pd.DataFrame({"Isolamento (3/66)": iso.existentes, **{n: serie(*k).existentes for k, n in REFERENCIAS.items()}})
mostrar(comp.iloc[::12])

fig, ax = plt.subplots(figsize=(11, 4.5))
(comp.div(comp.iloc[0]) * 100).plot(ax=ax, lw=2)
ax.set(title="Leitos existentes — índice (jan/2007 = 100)", ylabel="Índice", xlabel="Competência")
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

## Exportação

In [ ]:
# [6] Salva a série do Brasil
saida = BASE.parent / "resultados" / "leitos" / "leitos_isolamento_brasil.csv"
out = iso.rename_axis("competencia").reset_index()
out["competencia"] = out["competencia"].dt.strftime("%Y-%m")
out.to_csv(saida, index=False)
print("Gravado:", saida, "|", len(out), "linhas")